In [28]:
import pandas as pd
import random
from torch.utils.data import DataLoader
from sentence_transformers.cross_encoder import CrossEncoder
from sentence_transformers.cross_encoder.evaluation import CEBinaryClassificationEvaluator
from sentence_transformers import InputExample
from head_tail import  extract_head_tail

In [31]:
def train_cross_encoder(train_csv_path, output_model_path):
    print("Loading training data...")
    df = pd.read_csv(train_csv_path)
    df = df.dropna(subset=['text'])
    texts = df['text'].tolist()

    # 1. Extract Edges
    heads = []
    tails = []
    for text in texts:
        h, t = extract_head_tail(text, 150)
        if not h or not t:
            continue
        heads.append(h)
        tails.append(t)

    # 2. Generate Explicit Positive and Negative Pairs
    train_examples = []
    num_pages = len(heads)

    for i in range(num_pages - 1):
        if not heads[i+1] or not tails[i]:
            continue
        train_examples.append(InputExample(texts=[tails[i], heads[i+1]], label=1.0))

        hard_indices = []

        # Page_{i-1}
        if i - 1 >= 0:
            hard_indices.append(i - 1)

        # Page_{i+2}
        if i + 2 < num_pages:
            hard_indices.append(i + 2)

        # Page_{i+3}
        if i + 3 < num_pages:
            hard_indices.append(i + 3)

        for idx in hard_indices:
            neg = heads[idx]
            if neg:
                train_examples.append(InputExample(
                    texts=[tails[i], neg],
                    label=0.0
                ))

        random_idx = random.randint(0, num_pages - 1)
        if random_idx not in [i+1] + hard_indices:
            neg = heads[random_idx]
            if neg:
                train_examples.append(InputExample(
                    texts=[tails[i], neg],
                    label=0.0
                ))

    # ✅ Clean AFTER generation, now train_examples is actually populated
    cleaned_examples = []
    for ex in train_examples:
        t1, t2 = ex.texts
        label = ex.label
        if (
            isinstance(t1, str) and isinstance(t2, str)
            and t1.strip() != "" and t2.strip() != ""
            and label is not None
        ):
            cleaned_examples.append(ex)

    train_examples = cleaned_examples
    random.shuffle(train_examples)
    print(f"Generated {len(train_examples)} total pairs for Cross-Encoder.")
    
    # Split a tiny portion off for evaluation during training
    train_split = int(len(train_examples) * 0.9)
    train_data = train_examples[:train_split]
    eval_data = train_examples[train_split:]
    
    # 3. Load Base Model (DeBERTa-v3 is excellent for NLI and sequence classification)
    # Using 'small' to fit in memory, but 'base' or 'large' will yield higher Kaggle scores
    print("Loading DeBERTa model...")
    model = CrossEncoder('microsoft/deberta-v3-large', num_labels=1)
    
    # 4. Setup Dataloader and Evaluator
    train_dataloader = DataLoader(train_data, shuffle=True, batch_size=8)
    evaluator = CEBinaryClassificationEvaluator.from_input_examples(eval_data, name='train-eval')
    
    print(f"Train size: {len(train_data)}, Eval size: {len(eval_data)}")

    print(train_data[-2:])

    # NaN != NaN is True in Python — catches float NaN that slips past isinstance()
    bad = [
        (i, ex.texts, ex.label)
        for i, ex in enumerate(train_data + eval_data)
        if any(t != t for t in ex.texts) or ex.label != ex.label
    ]
    if bad:
        print(f"Found {len(bad)} bad examples: {bad[:3]}")
    else:
        print("Data looks clean.")

    # 5. Train Model
    print("Starting Cross-Encoder training...")
    model.fit(
        train_dataloader=train_dataloader,
        evaluator=evaluator,
        epochs=3,
        warmup_steps=100,
        output_path=output_model_path,
        show_progress_bar=True,
        use_amp=False,
        max_grad_norm=1.0, 
        weight_decay=0.01,                            # NEW: Regularize weights
        optimizer_params={'lr': 3e-6, 'eps': 1e-6}    # NEW: Microscopic LR + Epsilon fix
    )
    print(f"Saving complete model to {output_model_path}...")
    model.save(output_model_path)
    print("Training and saving complete!")

In [32]:
train_cross_encoder('./data/Mysterious_Affair_at_Styles_Train_Data.csv', './models/fine_tuned_cross_encoder_large')

Loading training data...
Generated 1022 total pairs for Cross-Encoder.
Loading DeBERTa model...


config.json:   0%|          | 0.00/580 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/874M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/874M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

...

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Step,Training Loss


Saving complete model to ./models/fine_tuned_cross_encoder_large...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training and saving complete!


In [35]:
import numpy as np
def solve_narrative_order_ce(input_csv, output_csv, model_path='./models/fine_tuned_cross_encoder_large', beam_width=3):
    print(f"Processing {input_csv}...")
    
    # Load dataset
    df = pd.read_csv(input_csv)
    page_ids = df['page'].tolist()
    texts = df['text'].tolist()
    num_pages = len(page_ids)
    
    # Apply Edge Extraction to all pages
    print("1. Extracting 60-word edges (Heads and Tails)...")
    heads, tails = zip(*[extract_head_tail(t) for t in texts])
    
    # ==========================================
    # STEP 2 & 3: Cross-Encoder Prediction
    # ==========================================
    print(f"2. Loading custom Cross-Encoder from '{model_path}'...")
    try:
        model = CrossEncoder(model_path)
    except Exception as e:
        print(f"Error loading model: {e}")
        print("Make sure you have run the training script first and the path is correct.")
        return

    print("3. Building N x N Transition Matrix (This may take a moment)...")
    transition_matrix = np.zeros((num_pages, num_pages))
    
    for i in range(num_pages):
        # Create pairs of (Tail i, Head j) for all j
        pairs = [[tails[i], heads[j]] for j in range(num_pages)]
        
        # The model outputs a probability/logit score for all pairs at once
        scores = model.predict(pairs)
        transition_matrix[i] = scores
        
        # Optional: Print progress for large books
        if (i + 1) % 20 == 0:
            print(f"   Scored transitions for {i + 1}/{num_pages} pages...")
            
    # Prevent a page from linking to itself by setting the diagonal to -infinity
    np.fill_diagonal(transition_matrix, -np.inf)


    print(f"4. Running Beam Search (beam width = {beam_width})...")

    max_incoming = np.max(transition_matrix, axis=0)
    start_candidates = np.argsort(max_incoming)[:beam_width]

    beams = [([start], 0.0) for start in start_candidates]

    for step in range(num_pages - 1):
        new_beams = []

        for seq, score in beams:
            last = seq[-1]

            for nxt in range(num_pages):
                if nxt in seq:
                    continue

                new_score = score + transition_matrix[last][nxt]
                new_seq = seq + [nxt]
                new_beams.append((new_seq, new_score))

        # Keep top K sequences
        new_beams = sorted(new_beams, key=lambda x: x[1], reverse=True)[:beam_width]
        beams = new_beams

    best_sequence = beams[0][0]
    
    # ==========================================
    # STEP 4: Greedy Traversal
    # ==========================================
    # print("4. Executing Greedy Traversal...")
    
    # # Find Page 1: The page that is least likely to come AFTER any other page
    # # We look at the maximum score each page gets as a "Head"
    # max_incoming_scores = np.max(transition_matrix, axis=0)
    # current_page_idx = np.argmin(max_incoming_scores)
    
    # ordered_indices = [current_page_idx]
    # visited = set([current_page_idx])
    
    # # Build the chain
    # for _ in range(num_pages - 1):
    #     # Get transition scores from the current page's tail to all other heads
    #     scores_from_current = transition_matrix[current_page_idx].copy()
        
    #     # Mask out pages we've already placed in the sequence
    #     for v in visited:
    #         scores_from_current[v] = -np.inf
            
    #     # Pick the best match
    #     next_page_idx = np.argmax(scores_from_current)
        
    #     # Add to our sequence and update state
    #     ordered_indices.append(next_page_idx)
    #     visited.add(next_page_idx)
    #     current_page_idx = next_page_idx

    # ==========================================
    # Formatting and Saving Submission
    # ==========================================
    print(f"Saving results to {output_csv}...")
    submission_data = {
        'original_page': range(1, num_pages + 1), # 1-based indexing required by evaluation
        'shuffled_page': [page_ids[i] for i in best_sequence]
    }
    
    pd.DataFrame(submission_data).to_csv(output_csv, index=False)
    print("Done!\n")

In [36]:
solve_narrative_order_ce('./data/BookA_test.csv', './submissions/BookA_cross_encoder_large.csv')
solve_narrative_order_ce('./data/BookB_test.csv', './submissions/BookB_cross_encoder_large.csv')

Processing ./data/BookA_test.csv...
1. Extracting 60-word edges (Heads and Tails)...
2. Loading custom Cross-Encoder from './models/fine_tuned_cross_encoder_large'...


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

3. Building N x N Transition Matrix (This may take a moment)...
   Scored transitions for 20/147 pages...
   Scored transitions for 40/147 pages...
   Scored transitions for 60/147 pages...
   Scored transitions for 80/147 pages...
   Scored transitions for 100/147 pages...
   Scored transitions for 120/147 pages...
   Scored transitions for 140/147 pages...
4. Running Beam Search (beam width = 3)...
Saving results to ./submissions/BookA_cross_encoder_large.csv...
Done!

Processing ./data/BookB_test.csv...
1. Extracting 60-word edges (Heads and Tails)...
2. Loading custom Cross-Encoder from './models/fine_tuned_cross_encoder_large'...


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

3. Building N x N Transition Matrix (This may take a moment)...
   Scored transitions for 20/57 pages...
   Scored transitions for 40/57 pages...
4. Running Beam Search (beam width = 3)...
Saving results to ./submissions/BookB_cross_encoder_large.csv...
Done!

